# Autoencoder Experiments: From Theory to Practice

**Practical companion to the Autoencoder Architectures lecture.**

This notebook walks through four progressively richer autoencoder families, each section building on the previous:

| # | Model | Key idea | Dataset (main → homework) |
|---|-------|----------|--------------------------|
| 1 | **Classical AE** | Bottleneck compression | MNIST → FashionMNIST, CIFAR-10 |
| 2 | **Denoising AE** | Reconstruct clean from corrupt | MNIST → FashionMNIST, CIFAR-10 |
| 3 | **Variational AE** | Probabilistic latent space | MNIST → FashionMNIST, CIFAR-10 |
| 4 | **Sparse AE** | L1 penalty on activations | MNIST → FashionMNIST, CIFAR-10 |

---

### Environment setup (run once)

```bash
# Install all dependencies with uv (from the project root):
uv sync

# Launch Jupyter:
uv run jupyter notebook ae_experiments.ipynb
```

All heavy imports live in the next cell — run it first.

In [ ]:
"""
=============================================================
 Cell 1 — Imports, reproducibility, and device selection
=============================================================
Run this cell first.  Every subsequent cell depends on it.
"""

import math
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as T
from torchvision.datasets import MNIST, FashionMNIST, CIFAR10

from tqdm.notebook import tqdm


# ── Reproducibility ───────────────────────────────────────────────────────────
# Fixing all random seeds makes results identical across runs.
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)


# ── Device selection ──────────────────────────────────────────────────────────
# Priority: Apple MPS (M-series chip) → NVIDIA CUDA → CPU
# MPS is Apple's Metal Performance Shaders GPU backend — very fast on M4.
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

# MPS note: pin_memory and certain num_workers settings are incompatible with MPS.
PIN_MEMORY  = (DEVICE.type == "cuda")   # only helpful for CUDA
NUM_WORKERS = 0                          # 0 is safest on macOS (avoids fork issues)

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")
print(f"CUDA    : {torch.cuda.is_available()}")
print(f"MPS     : {torch.backends.mps.is_available()}")

In [ ]:
"""
=============================================================
 Cell 2 — Shared visualisation helpers (used by all parts)
=============================================================
"""

def imshow_grid(images, titles=None, n_cols=8, scale=1.5, cmap="gray", suptitle=None):
    """
    Display a list or batch of images in a tidy grid.

    Parameters
    ----------
    images   : list of tensors (C,H,W) or (H,W), or a single (N,C,H,W) tensor
    titles   : optional list of strings (one per image)
    n_cols   : number of columns
    scale    : size of each cell in inches
    cmap     : colormap — 'gray' for grayscale, None lets matplotlib decide for RGB
    suptitle : optional overall title
    """
    if isinstance(images, torch.Tensor):
        images = [images[i] for i in range(len(images))]

    n      = len(images)
    n_rows = math.ceil(n / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * scale, n_rows * scale))
    axes = np.array(axes).flatten()

    for i, ax in enumerate(axes):
        if i < n:
            img = images[i]
            if isinstance(img, torch.Tensor):
                img = img.detach().cpu()
                # (C,H,W) → (H,W,C) or (H,W) for single-channel
                if img.ndim == 3:
                    img = img.permute(1, 2, 0).numpy()
                    if img.shape[2] == 1:
                        img = img.squeeze(2)
                else:
                    img = img.numpy()
            img = np.clip(img, 0.0, 1.0)
            ax.imshow(img, cmap=cmap if img.ndim == 2 else None)
            if titles:
                ax.set_title(str(titles[i]), fontsize=8)
        ax.axis("off")

    if suptitle:
        plt.suptitle(suptitle, fontsize=12, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()


def plot_losses(history: dict, title: str = "Training curves"):
    """
    Plot one or more loss curves from a dict of {label: [values]}.

    Example:
        plot_losses({"train": train_losses, "test": test_losses})
    """
    plt.figure(figsize=(7, 3))
    for label, vals in history.items():
        plt.plot(vals, label=label, linewidth=2)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def compare_rows(row_dict: dict, n_images: int = 8, img_shape=(1, 28, 28), suptitle=""):
    """
    Show multiple rows of images (e.g. original / noisy / reconstructed).

    Parameters
    ----------
    row_dict  : OrderedDict-like {row_label: tensor_batch}
                Each tensor_batch has shape (N, *) where * will be reshaped to img_shape
    n_images  : how many columns to show
    img_shape : (C, H, W) of each image
    """
    n_rows = len(row_dict)
    fig, axes = plt.subplots(n_rows, n_images,
                             figsize=(n_images * 1.4, n_rows * 1.6))
    if n_rows == 1:
        axes = axes[np.newaxis, :]   # ensure 2-D axes array

    for row_idx, (label, batch) in enumerate(row_dict.items()):
        for col_idx in range(n_images):
            img = batch[col_idx].detach().cpu()
            img = img.reshape(img_shape)
            if img_shape[0] == 1:
                axes[row_idx, col_idx].imshow(img.squeeze(0).numpy().clip(0, 1), cmap="gray")
            else:
                axes[row_idx, col_idx].imshow(img.permute(1, 2, 0).numpy().clip(0, 1))
            axes[row_idx, col_idx].axis("off")
        axes[row_idx, 0].set_ylabel(label, fontsize=10, fontweight="bold", labelpad=8)

    if suptitle:
        plt.suptitle(suptitle, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

print("Helpers loaded.")

In [ ]:
"""
=============================================================
 Cell 3 — Load datasets (MNIST · FashionMNIST · CIFAR-10)
=============================================================
torchvision downloads them automatically to ./data/ on first run.
Subsequent runs use the cached copy.
"""

DATA_DIR   = "./data"
BATCH_SIZE = 512    # works well on M4 unified memory; halve if you see OOM errors


# ── Transform pipelines ───────────────────────────────────────────────────────
# ToTensor()  : PIL image (H,W) uint8 [0,255]  →  float32 tensor (C,H,W) [0,1]
# Normalize() : optional — we skip it here so pixel values stay in [0,1],
#               which matches the Sigmoid output of all our decoders.

grayscale_tf = T.Compose([T.ToTensor()])        # → (1, 28, 28), values ∈ [0,1]
cifar_tf     = T.Compose([T.ToTensor()])        # → (3, 32, 32), values ∈ [0,1]


# ── MNIST ─────────────────────────────────────────────────────────────────────
mnist_train = MNIST(DATA_DIR, train=True,  transform=grayscale_tf, download=True)
mnist_test  = MNIST(DATA_DIR, train=False, transform=grayscale_tf, download=True)

mnist_train_loader = DataLoader(mnist_train, batch_size=BATCH_SIZE,
                                shuffle=True,  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
mnist_test_loader  = DataLoader(mnist_test,  batch_size=BATCH_SIZE,
                                shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


# ── FashionMNIST (homework) ───────────────────────────────────────────────────
fmnist_train = FashionMNIST(DATA_DIR, train=True,  transform=grayscale_tf, download=True)
fmnist_test  = FashionMNIST(DATA_DIR, train=False, transform=grayscale_tf, download=True)

fmnist_train_loader = DataLoader(fmnist_train, batch_size=BATCH_SIZE,
                                  shuffle=True,  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
fmnist_test_loader  = DataLoader(fmnist_test,  batch_size=BATCH_SIZE,
                                  shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

FMNIST_CLASSES = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
                  "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]


# ── CIFAR-10 (homework) ───────────────────────────────────────────────────────
cifar_train = CIFAR10(DATA_DIR, train=True,  transform=cifar_tf, download=True)
cifar_test  = CIFAR10(DATA_DIR, train=False, transform=cifar_tf, download=True)

cifar_train_loader = DataLoader(cifar_train, batch_size=256,   # smaller: 3×32×32 is larger
                                 shuffle=True,  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
cifar_test_loader  = DataLoader(cifar_test,  batch_size=256,
                                 shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

CIFAR_CLASSES = ["airplane","automobile","bird","cat","deer",
                 "dog","frog","horse","ship","truck"]


# ── Quick preview ─────────────────────────────────────────────────────────────
print(f"MNIST        train={len(mnist_train):,}  test={len(mnist_test):,}   shape={mnist_train[0][0].shape}")
print(f"FashionMNIST train={len(fmnist_train):,}  test={len(fmnist_test):,}   shape={fmnist_train[0][0].shape}")
print(f"CIFAR-10     train={len(cifar_train):,}  test={len(cifar_test):,}  shape={cifar_train[0][0].shape}")

sample_mnist  = [mnist_train[i][0]  for i in range(16)]
sample_fmnist = [fmnist_train[i][0] for i in range(16)]
sample_cifar  = [cifar_train[i][0]  for i in range(16)]

imshow_grid(sample_mnist,  titles=[mnist_train[i][1]  for i in range(16)],
            suptitle="MNIST samples")
imshow_grid(sample_fmnist, titles=[FMNIST_CLASSES[fmnist_train[i][1]] for i in range(16)],
            suptitle="FashionMNIST samples")
imshow_grid(sample_cifar,  titles=[CIFAR_CLASSES[cifar_train[i][1]] for i in range(16)],
            cmap=None, suptitle="CIFAR-10 samples")

---
## Part 1 — Classical Autoencoder

### What problem does it solve?

We want to learn a compact, informative representation of data **without labels**. An autoencoder achieves this by training a neural network to be its own teacher: the target output is the input itself.

### Architecture

```
x ──► Encoder ──► z (latent code) ──► Decoder ──► x̂ (reconstruction)
784         f(·)       2–32                g(·)         784
```

The **bottleneck** (latent space) is the key constraint. Because $\dim(z) \ll \dim(x)$, the network *cannot* copy the input and must instead learn to capture the most important structure.

### Loss function — Mean Squared Error (MSE)

$$\mathcal{L}(x,\hat{x}) = \frac{1}{N} \sum_{i=1}^{N} \|x_i - \hat{x}_i\|^2 = \frac{1}{N} \sum_{i=1}^{N} \|x_i - g(f(x_i))\|^2$$

Where:
- $f: \mathbb{R}^D \to \mathbb{R}^d$ is the **encoder** (compresses, $d \ll D$)
- $g: \mathbb{R}^d \to \mathbb{R}^D$ is the **decoder** (expands back)
- $z = f(x)$ is the **latent code** (the bottleneck representation)

### Connection to PCA

If both encoder and decoder are **linear** (no activation functions) and loss is MSE, the autoencoder's latent space spans exactly the same subspace as **Principal Component Analysis (PCA)**. Non-linear activations (ReLU) give us non-linear dimensionality reduction — strictly more powerful than PCA.

### Architecture we'll train

| Layer | Size | Notes |
|-------|------|-------|
| Input | 784 | flattened 28×28 |
| Linear → BatchNorm → ReLU | 256 | encoder layer 1 |
| Linear → BatchNorm → ReLU | 64  | encoder layer 2 |
| **Linear** | **`latent_dim`** | bottleneck |
| Linear → BatchNorm → ReLU | 64  | decoder layer 1 |
| Linear → BatchNorm → ReLU | 256 | decoder layer 2 |
| Linear → **Sigmoid** | 784 | output in [0,1] |

We train with `latent_dim=2` first (for beautiful 2D visualisation) and then with `latent_dim=16` (for better reconstruction quality).

In [ ]:
"""
=============================================================
 Cell 5 — Classical Autoencoder model
=============================================================
We define Encoder, Decoder, and Autoencoder as three separate
nn.Module classes for clarity.  They are reusable: the Denoising
AE in Part 2 uses the same backbone.
"""

class Encoder(nn.Module):
    """
    Compresses x ∈ ℝ^input_dim  →  z ∈ ℝ^latent_dim.

    BatchNorm1d is placed BEFORE ReLU (BN → ReLU is the standard order).
    It normalises each mini-batch, which:
      • speeds up training (allows higher LR)
      • reduces sensitivity to weight initialisation
      • acts as a mild regulariser

    The last linear layer has NO activation — latent codes should be
    unconstrained real numbers.
    """
    def __init__(self, input_dim: int = 784, latent_dim: int = 16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Linear(64, latent_dim),   # ← bottleneck; no activation
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x : (batch, C, H, W) or already (batch, D)
        return self.net(x.flatten(start_dim=1))   # flatten everything except batch dim


class Decoder(nn.Module):
    """
    Expands z ∈ ℝ^latent_dim  →  x̂ ∈ ℝ^output_dim.

    Sigmoid at the end maps outputs to [0, 1], matching the
    normalised pixel range produced by ToTensor().
    """
    def __init__(self, latent_dim: int = 16, output_dim: int = 784):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Linear(64, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            nn.Linear(256, output_dim),
            nn.Sigmoid(),   # ← squash to [0,1] so MSE has a clean scale
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)   # output shape: (batch, output_dim)


class Autoencoder(nn.Module):
    """
    Full model: Encoder → Bottleneck → Decoder.

    Forward returns x̂ (reconstruction).
    Auxiliary methods encode() and decode() let us inspect the
    latent space independently (useful for visualisation).
    """
    def __init__(self, input_dim: int = 784, latent_dim: int = 16):
        super().__init__()
        self.encoder = Encoder(input_dim, latent_dim)
        self.decoder = Decoder(latent_dim, input_dim)
        self.latent_dim = latent_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))

    @torch.no_grad()
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Project x into latent space (inference-only, no gradient)."""
        return self.encoder(x)

    @torch.no_grad()
    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """Decode latent code z back to image space."""
        return self.decoder(z)


# ── Quick sanity check ────────────────────────────────────────────────────────
_model = Autoencoder(latent_dim=2)
_x     = torch.randn(4, 1, 28, 28)          # fake batch of 4 images
_xhat  = _model(_x)
print(f"Input  shape : {_x.shape}")
print(f"Output shape : {_xhat.shape}  (should be same as input when flattened)")
print(f"Encoder output shape: {_model.encode(_x).shape}  ← the 2-D latent code")

# Count parameters
n_params = sum(p.numel() for p in _model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {n_params:,}")

In [ ]:
"""
=============================================================
 Cell 6 — Generic AE training loop
=============================================================
This function is reused in Part 2 (Denoising AE) via the
optional `corrupt_fn` argument.
"""

def train_ae(
    model,
    train_loader,
    test_loader,
    n_epochs:    int   = 20,
    lr:          float = 1e-3,
    device             = DEVICE,
    corrupt_fn         = None,   # if given: corrupt input before encoding
    verbose:     bool  = True,
) -> dict:
    """
    Train an Autoencoder with MSE reconstruction loss.

    Parameters
    ----------
    model        : nn.Module with a forward(x) → x̂ interface
    train_loader : DataLoader for training set
    test_loader  : DataLoader for validation / test set
    n_epochs     : number of full passes over the training data
    lr           : Adam learning rate
    device       : torch.device (cpu / cuda / mps)
    corrupt_fn   : optional callable(x) → x̃  (for Denoising AE in Part 2)
                   If provided, the model sees x̃ but loss is computed vs clean x.
    verbose      : whether to print epoch summaries

    Returns
    -------
    dict with keys "train" and "test" containing per-epoch loss lists.
    """
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # ReduceLROnPlateau: halve LR if test loss hasn't improved for 3 epochs.
    # This often lets us train a bit longer without overfitting.
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3,
    )

    history = {"train": [], "test": []}

    for epoch in range(1, n_epochs + 1):
        # ── Train ──────────────────────────────────────────────────────────────
        model.train()
        running_loss = 0.0

        for x_clean, _ in tqdm(train_loader, desc=f"Epoch {epoch}/{n_epochs}", leave=False):
            x_clean = x_clean.to(device)

            # Optionally corrupt the input (Denoising AE uses this)
            x_in = corrupt_fn(x_clean) if corrupt_fn else x_clean

            # Forward pass: model sees x_in, we compare output to clean x
            x_hat = model(x_in)                           # (batch, 784)
            x_tgt = x_clean.flatten(start_dim=1)         # (batch, 784)

            loss = F.mse_loss(x_hat, x_tgt)              # mean over pixels & batch

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)
        history["train"].append(train_loss)

        # ── Evaluate ───────────────────────────────────────────────────────────
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for x_clean, _ in test_loader:
                x_clean = x_clean.to(device)
                x_in    = corrupt_fn(x_clean) if corrupt_fn else x_clean
                x_hat   = model(x_in)
                x_tgt   = x_clean.flatten(start_dim=1)
                test_loss += F.mse_loss(x_hat, x_tgt).item()
        test_loss /= len(test_loader)
        history["test"].append(test_loss)

        scheduler.step(test_loss)   # adjust LR based on test loss

        if verbose and (epoch % 5 == 0 or epoch == 1):
            lr_now = optimizer.param_groups[0]["lr"]
            print(f"  Epoch {epoch:3d} | train={train_loss:.5f}  "
                  f"test={test_loss:.5f}  lr={lr_now:.2e}")

    return history

print("train_ae() defined.")

In [ ]:
"""
=============================================================
 Cell 7 — Train Classical AE on MNIST (latent_dim=2 and 16)
=============================================================

We train two models:
  • ae2d  — 2-dimensional latent space  → perfect for 2D visualisation
  • ae16d — 16-dimensional latent space → much better reconstruction
"""

# ── Model 1: 2-D latent (for latent space plot) ───────────────────────────────
print("=" * 55)
print("Training AE with latent_dim=2  (2-D latent space)")
print("=" * 55)
set_seed(42)
ae2d = Autoencoder(latent_dim=2)

history_ae2d = train_ae(
    ae2d,
    mnist_train_loader,
    mnist_test_loader,
    n_epochs=25,
    lr=1e-3,
)

plot_losses(history_ae2d, title="Classical AE (latent_dim=2) — MNIST")

# ── Model 2: 16-D latent (for quality reconstruction) ────────────────────────
print("\n" + "=" * 55)
print("Training AE with latent_dim=16 (higher quality)")
print("=" * 55)
set_seed(42)
ae16d = Autoencoder(latent_dim=16)

history_ae16d = train_ae(
    ae16d,
    mnist_train_loader,
    mnist_test_loader,
    n_epochs=25,
    lr=1e-3,
)

plot_losses(history_ae16d, title="Classical AE (latent_dim=16) — MNIST")

In [ ]:
"""
=============================================================
 Cell 8 — Visualise reconstructions & 2-D latent space
=============================================================
"""

# ── 1. Reconstruction quality comparison ─────────────────────────────────────
# Pull one batch from the test set (we do NOT pass it to train_ae, so it's truly unseen).
x_test_batch, labels_batch = next(iter(mnist_test_loader))
x_test_8 = x_test_batch[:8].to(DEVICE)

ae2d.eval();  ae16d.eval()

with torch.no_grad():
    xhat_2d  = ae2d(x_test_8).reshape(-1, 1, 28, 28)
    xhat_16d = ae16d(x_test_8).reshape(-1, 1, 28, 28)

compare_rows(
    {
        "Original"         : x_test_8,
        "Recon  (dim=2)"   : xhat_2d,
        "Recon  (dim=16)"  : xhat_16d,
    },
    n_images=8,
    img_shape=(1, 28, 28),
    suptitle="Classical AE — reconstruction quality vs latent dimension",
)

# ── 2. 2-D latent space ───────────────────────────────────────────────────────
# Encode the ENTIRE test set and colour each point by its true digit label.
# A good latent space shows well-separated digit clusters.

all_z, all_y = [], []
ae2d.eval()
with torch.no_grad():
    for xb, yb in mnist_test_loader:
        z = ae2d.encoder(xb.to(DEVICE)).cpu()  # use encoder directly (no grad)
        all_z.append(z)
        all_y.append(yb)

all_z = torch.cat(all_z).numpy()   # (10000, 2)
all_y = torch.cat(all_y).numpy()   # (10000,)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(all_z[:, 0], all_z[:, 1],
                      c=all_y, cmap="tab10", s=3, alpha=0.7)
cb = plt.colorbar(scatter, ticks=range(10))
cb.set_label("Digit class")
plt.xlabel("Latent dimension 1  (z₁)")
plt.ylabel("Latent dimension 2  (z₂)")
plt.title("Classical AE — 2-D Latent Space (MNIST test set)\n"
          "Each dot = one image encoded to 2 numbers", fontsize=11)
plt.tight_layout()
plt.show()

# ── 3. Latent space grid ───────────────────────────────────────────────────────
# Walk a uniform grid through the 2-D latent space and decode each point.
# This shows what each region of the latent space "means" to the decoder.

def decode_grid(model, grid_range=3.5, n=18):
    """Decode a uniform (n×n) grid in 2-D latent space."""
    model.eval()
    vals   = torch.linspace(-grid_range, grid_range, n)
    # meshgrid: z1 varies along columns, z2 along rows (so image looks upright)
    zz1, zz2 = torch.meshgrid(vals, vals, indexing="xy")
    z_grid = torch.stack([zz1.flatten(), zz2.flatten()], dim=1).to(DEVICE)

    with torch.no_grad():
        imgs = model.decoder(z_grid).cpu()   # (n*n, 784)

    imgs = imgs.reshape(n, n, 28, 28)         # (row, col, H, W)

    # Stitch tiles into one big image
    canvas = imgs.permute(0, 2, 1, 3).reshape(n * 28, n * 28).numpy()
    return canvas

canvas = decode_grid(ae2d)
plt.figure(figsize=(9, 9))
plt.imshow(canvas, cmap="gray", origin="upper")
plt.axis("off")
plt.title("Latent Space Grid Traversal — each tile is a decoded z-point\n"
          "(left→right: z₁ from −3.5 to +3.5;  top→bottom: z₂ from −3.5 to +3.5)",
          fontsize=10)
plt.tight_layout()
plt.show()

---
### 🏠 Homework — Part 1: Classical Autoencoder

#### Task A — FashionMNIST
Run the **same model and training code** on FashionMNIST (loaders are already loaded as `fmnist_train_loader` / `fmnist_test_loader`).

```python
set_seed(0)
ae_fmnist = Autoencoder(latent_dim=2)
history_fmnist = train_ae(ae_fmnist, fmnist_train_loader, fmnist_test_loader, n_epochs=25)
plot_losses(history_fmnist, "AE latent_dim=2 — FashionMNIST")
# Then visualise reconstructions and 2-D latent space (same code as Cell 8)
# Q: Are the 10 clothing clusters more/less separated than MNIST digits?  Why?
```

#### Task B — CIFAR-10 with Convolutional AE

Flat linear layers don't respect spatial structure. For colour images a **Convolutional AE** works much better. Complete and train the skeleton below:

```python
class ConvEncoder(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.conv = nn.Sequential(
            # 3×32×32  →  32×16×16
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            # 32×16×16 →  64×8×8
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            # 64×8×8  → 128×4×4
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc = nn.Linear(128 * 4 * 4, latent_dim)

    def forward(self, x):
        return self.fc(self.conv(x).flatten(1))


class ConvDecoder(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128 * 4 * 4)
        self.deconv = nn.Sequential(
            # 4→8
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            # 8→16
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            # 16→32
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, z):
        return self.deconv(self.fc(z).view(-1, 128, 4, 4))


class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.encoder = ConvEncoder(latent_dim)
        self.decoder = ConvDecoder(latent_dim)

    def forward(self, x):
        return self.decoder(self.encoder(x))


# Train and evaluate
set_seed(0)
ae_cifar  = ConvAutoencoder(latent_dim=64)
optimizer = torch.optim.Adam(ae_cifar.parameters(), lr=1e-3)

# ← Your training loop here! Adapt train_ae or write your own.
# Hint: ConvDecoder returns (batch, 3, 32, 32) so reshape accordingly.
# Q: How does reconstruction look on test images after 20 epochs?
```

#### Reflection questions (answer in a markdown cell below)
1. How does reconstruction quality change as you increase `latent_dim` from 2 → 8 → 32 → 128?
2. Why does the 2-D latent space of MNIST look like a "banana" rather than a circle?
3. What happens if `latent_dim` ≥ `input_dim`? (Try it! Use `latent_dim=800` on MNIST.)

---
## Part 2 — Denoising Autoencoder (DAE)

### Motivation

A classical AE with a large enough bottleneck can learn a near-perfect identity mapping — it just memorises every detail, including noise. A **Denoising AE** adds deliberate corruption to the *input* but keeps the *target* clean. The model is therefore forced to understand the true structure of the data (the manifold) and ignore surface noise.

### The key trick

$$\tilde{x} = \text{corrupt}(x) \qquad \hat{x} = g(f(\tilde{x})) \qquad \mathcal{L} = \|x - \hat{x}\|^2$$

The loss is computed against the **original** $x$, not the noisy $\tilde{x}$. This prevents the model from simply copying corruption patterns.

### Why it works — manifold intuition

Real-world data lives on a low-dimensional **manifold** embedded in high-dimensional pixel space. A noisy image $\tilde{x}$ is a point slightly *off* the manifold. Training a DAE to denoise teaches the model the direction from any nearby off-manifold point *back to* the manifold — i.e., it learns the manifold's shape.

### Three corruption strategies we'll use

| Corruption | Formula | Intuition |
|------------|---------|-----------|
| Gaussian noise | $\tilde{x} = x + \epsilon,\ \epsilon \sim \mathcal{N}(0, \sigma^2)$ | Adds continuous random perturbations |
| Masking noise | $\tilde{x}_i = x_i \cdot m_i,\ m_i \sim \text{Bernoulli}(1-p)$ | Randomly zeros out pixels |
| Salt-and-pepper | set pixel to 0 or 1 with prob $p$ | Extreme outlier pixels |

Notice: the **model architecture is identical** to the classical AE. Only the *training procedure* changes.

In [ ]:
"""
=============================================================
 Cell 11 — Noise functions + visualise corruption types
=============================================================
All functions are written to work in-place on a GPU tensor
without copying it back to CPU (important for performance).
"""

def gaussian_noise(x: torch.Tensor, sigma: float = 0.3) -> torch.Tensor:
    """
    Add Gaussian (normal) noise to x and clip to [0, 1].

    x̃ = clip(x + ε,  0, 1)    where  ε ~ N(0, σ²)

    sigma controls noise strength:
      0.1 → mild blur, digits still readable
      0.3 → moderate (used here)
      0.5 → heavily degraded
    """
    noise = sigma * torch.randn_like(x)          # same shape & device as x
    return (x + noise).clamp(0.0, 1.0)


def masking_noise(x: torch.Tensor, drop_prob: float = 0.4) -> torch.Tensor:
    """
    Set each pixel to 0 independently with probability drop_prob.

    x̃_i = x_i * m_i    where  m_i ~ Bernoulli(1 - drop_prob)

    This mimics the spirit of BERT's [MASK] token in language models.
    The model must infer missing pixels from context.
    """
    # torch.rand_like gives uniform [0,1]; pixels where rand > drop_prob are kept
    mask = (torch.rand_like(x) > drop_prob).float()
    return x * mask


def salt_pepper_noise(x: torch.Tensor, noise_prob: float = 0.1) -> torch.Tensor:
    """
    Salt-and-pepper noise: each pixel is set to 0 (pepper) or 1 (salt)
    with equal probability noise_prob/2, otherwise left unchanged.

    Models must learn to detect and fix extreme outlier pixels.
    """
    noisy = x.clone()
    uniform = torch.rand_like(x)
    noisy[uniform < noise_prob / 2]                           = 0.0  # pepper
    noisy[(uniform >= noise_prob / 2) & (uniform < noise_prob)] = 1.0  # salt
    return noisy


# ── Visualise all three corruption types side-by-side ────────────────────────
x_vis = mnist_test_loader.dataset[0][0].unsqueeze(0)   # shape: (1, 1, 28, 28)

compare_rows(
    {
        "Clean original"       : x_vis.expand(8, -1, -1, -1),
        "Gaussian (σ=0.3)"     : gaussian_noise(x_vis.expand(8, -1, -1, -1), sigma=0.3),
        "Masking (p=0.4)"      : masking_noise(x_vis.expand(8, -1, -1, -1), drop_prob=0.4),
        "Salt-&-pepper (p=0.1)": salt_pepper_noise(x_vis.expand(8, -1, -1, -1), noise_prob=0.1),
    },
    n_images=8,
    img_shape=(1, 28, 28),
    suptitle="Three corruption strategies applied to the same image",
)

In [ ]:
"""
=============================================================
 Cell 12 — Train DAE on MNIST + visualise denoising results
=============================================================
We reuse the exact same Autoencoder class and train_ae() function
from Part 1 — the only difference is passing corrupt_fn=gaussian_noise.
"""

# ── Train with Gaussian noise ─────────────────────────────────────────────────
print("Training Denoising AE (Gaussian noise σ=0.3) …")
set_seed(42)

# We use a larger latent_dim=32 because the DAE task is harder than plain reconstruction.
# More latent dimensions give the encoder more room to represent clean structure.
dae_gaussian = Autoencoder(latent_dim=32)

# The magic is this single argument: corrupt_fn=gaussian_noise
# train_ae() corrupts each batch BEFORE encoding, but computes loss against clean x.
history_dae_gauss = train_ae(
    dae_gaussian,
    mnist_train_loader,
    mnist_test_loader,
    n_epochs=25,
    lr=1e-3,
    corrupt_fn=lambda x: gaussian_noise(x, sigma=0.3),
)

plot_losses(history_dae_gauss, "DAE (Gaussian noise) — MNIST")

# ── Train with masking noise ──────────────────────────────────────────────────
print("\nTraining Denoising AE (Masking noise p=0.4) …")
set_seed(42)
dae_masking = Autoencoder(latent_dim=32)

history_dae_mask = train_ae(
    dae_masking,
    mnist_train_loader,
    mnist_test_loader,
    n_epochs=25,
    lr=1e-3,
    corrupt_fn=lambda x: masking_noise(x, drop_prob=0.4),
)

# ── Visualise: clean  /  noisy  /  denoised (Gaussian) /  denoised (Mask) ────
x_test_batch, _ = next(iter(mnist_test_loader))
x_clean = x_test_batch[:8].to(DEVICE)

x_noisy_gauss = gaussian_noise(x_clean, sigma=0.3)
x_noisy_mask  = masking_noise(x_clean,  drop_prob=0.4)

dae_gaussian.eval(); dae_masking.eval()
with torch.no_grad():
    xhat_gauss = dae_gaussian(x_noisy_gauss).reshape(-1, 1, 28, 28)
    xhat_mask  = dae_masking(x_noisy_mask).reshape(-1, 1, 28, 28)

compare_rows(
    {
        "Clean input"            : x_clean,
        "Noisy (Gaussian σ=0.3)" : x_noisy_gauss,
        "Denoised (Gaussian DAE)": xhat_gauss,
        "Noisy (Masking p=0.4)"  : x_noisy_mask,
        "Denoised (Masking DAE)" : xhat_mask,
    },
    n_images=8,
    img_shape=(1, 28, 28),
    suptitle="DAE results: the model recovers clean digits from heavily corrupted input",
)

# ── Quantitative comparison: DAE vs classical AE (no denoising training) ──────
# How well does a standard AE (ae16d) denoise test images it was NOT trained for?
ae16d.eval()
with torch.no_grad():
    xhat_ae_nondae = ae16d(x_noisy_gauss).reshape(-1, 1, 28, 28)

compare_rows(
    {
        "Noisy input"         : x_noisy_gauss,
        "Standard AE (no DAE)": xhat_ae_nondae,
        "DAE (trained on noise)": xhat_gauss,
    },
    n_images=8,
    img_shape=(1, 28, 28),
    suptitle="DAE vs standard AE on noisy inputs — DAE wins because it saw noise during training",
)

---
### 🏠 Homework — Part 2: Denoising Autoencoder

#### Task A — FashionMNIST DAE
Train a DAE on FashionMNIST with masking noise (`drop_prob=0.5`) — a harder corruption than Gaussian noise. Use `latent_dim=32`.

```python
set_seed(0)
dae_fmnist = Autoencoder(latent_dim=32)
history_dae_fmnist = train_ae(
    dae_fmnist, fmnist_train_loader, fmnist_test_loader,
    n_epochs=25,
    corrupt_fn=lambda x: masking_noise(x, drop_prob=0.5),
)
# Visualise: which clothing categories are hardest to recover?
```

#### Task B — CIFAR-10 DAE with Conv architecture
Adapt `ConvAutoencoder` from Part 1 homework as the DAE backbone for CIFAR-10.

```python
# 1. Instantiate ConvAutoencoder(latent_dim=64)
# 2. Write a training loop that:
#    - corrupts each batch with gaussian_noise(x, sigma=0.2)
#    - computes MSE against the CLEAN batch
# 3. Visualise a 3-row grid: original / noisy / denoised
```

#### Task C — Noise sensitivity experiment
Train three Gaussian DAEs on MNIST with σ ∈ {0.1, 0.3, 0.6}.
Plot the test MSE loss of each. What happens when you apply σ=0.6 noise to the model trained on σ=0.1?

#### Reflection questions
1. Why does the Masking DAE not just learn to output zero for masked pixels?
2. Training a DAE on σ=0.3 noise and then testing on σ=0.5 noise — does it generalise? Why or why not?
3. What is the relationship between a Masking DAE on images and BERT's Masked Language Modelling?

---
## Part 3 — Variational Autoencoder (VAE)

### Problem with the classical AE's latent space

A trained classical AE has a **discrete, irregular** latent space. If you sample a random point $z$ not close to a real encoded image, the decoder outputs garbage. You cannot use it as a **generative model**.

### VAE: encoding as a distribution, not a point

Instead of encoding $x$ to a single vector $z$, the VAE encoder outputs the **parameters of a Gaussian distribution**:

$$q_\phi(z \mid x) = \mathcal{N}(\mu_\phi(x),\; \sigma_\phi^2(x) \cdot I)$$

At training time we **sample** from this distribution: $z \sim q_\phi(z|x)$. At inference we use $\mu$ directly.

### The Reparameterization Trick

Sampling is not differentiable, so gradients cannot flow back through it. The fix:

$$z = \mu + \sigma \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

Now $z$ is a **deterministic** function of $\mu$ and $\sigma$ (the parameters we learn), plus some noise $\epsilon$ that has no learnable parameters. Gradients flow freely through $\mu$ and $\sigma$.

### Loss — Evidence Lower BOund (ELBO)

$$\mathcal{L}_{\text{VAE}} = \underbrace{\mathbb{E}_{q}[\log p_\theta(x|z)]}_{\text{Reconstruction}} - \beta \cdot \underbrace{D_{\text{KL}}\!\bigl(q_\phi(z|x)\,\|\,p(z)\bigr)}_{\text{Regularisation}}$$

The two terms are in tension:
- **Reconstruction** pushes $\mu$ away from 0 and $\sigma$ toward 0 (encode more information).
- **KL divergence** pulls $\mu$ back to 0 and $\sigma$ back to 1 (stay close to the prior $\mathcal{N}(0,I)$).

The KL term has a **closed-form** solution for Gaussians:

$$D_{\text{KL}}\!\bigl(\mathcal{N}(\mu, \sigma^2)\,\|\,\mathcal{N}(0,I)\bigr) = -\frac{1}{2}\sum_{j=1}^{d}\bigl(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\bigr)$$

The $\beta$ hyper-parameter lets us control the KL weight ($\beta=1$ is the original VAE; $\beta>1$ forces a more disentangled, structured latent space — called $\beta$-VAE).

### What we gain
Because the prior is a **standard Gaussian**, any point sampled from $\mathcal{N}(0,I)$ will decode to a realistic-looking image. The latent space is smooth and continuous — perfect for interpolation and generation.

In [ ]:
"""
=============================================================
 Cell 15 — VAE model (Encoder · Decoder · VAE)
=============================================================
"""

class VAEEncoder(nn.Module):
    """
    Encoder for the Variational Autoencoder.

    Unlike the classical encoder (one output), this has TWO output heads:
      • fc_mu      → mean  μ  of the posterior q(z|x)
      • fc_logvar  → log-variance  log σ²  (we use log for numerical stability;
                     exp of a large negative number is near 0, which is fine)

    The shared body extracts features; the two heads interpret them differently.
    """
    def __init__(self, input_dim: int = 784, hidden_dim: int = 256,
                 latent_dim: int = 2):
        super().__init__()
        # Shared feature extractor
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
        )
        self.fc_mu     = nn.Linear(64, latent_dim)   # mean
        self.fc_logvar = nn.Linear(64, latent_dim)   # log variance

    def forward(self, x: torch.Tensor):
        h      = self.shared(x.flatten(start_dim=1))
        mu     = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar   # both shape: (batch, latent_dim)


class VAEDecoder(nn.Module):
    """
    Decoder: same architecture as the classical decoder.
    During generation we can pass any z from N(0,I) and get a realistic image.
    """
    def __init__(self, latent_dim: int = 2, hidden_dim: int = 256,
                 output_dim: int = 784):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


class VAE(nn.Module):
    """
    Full Variational Autoencoder.

    forward() returns (x̂, μ, log σ²) — we need all three to compute the loss.
    encode()  returns only μ  (deterministic; used for visualisation).
    decode()  takes z ∈ ℝ^latent_dim  →  x̂ ∈ ℝ^784  (generation).
    sample()  draws a fresh z from the prior N(0,I) and decodes it.
    """
    def __init__(self, input_dim: int = 784, hidden_dim: int = 256,
                 latent_dim: int = 2):
        super().__init__()
        self.encoder    = VAEEncoder(input_dim, hidden_dim, latent_dim)
        self.decoder    = VAEDecoder(latent_dim, hidden_dim, input_dim)
        self.latent_dim = latent_dim

    def reparameterize(self, mu: torch.Tensor,
                        logvar: torch.Tensor) -> torch.Tensor:
        """
        Reparameterization trick:
            z = μ + σ · ε,    ε ~ N(0, I)

        During training  : sample stochastically (adds noise, acts as regulariser)
        During evaluation: return μ directly (deterministic, lower variance)

        self.training is automatically True/False via model.train() / model.eval().
        """
        if self.training:
            std     = torch.exp(0.5 * logvar)   # σ = exp(½ · log σ²)
            epsilon = torch.randn_like(std)       # ε ~ N(0, I), same shape as std
            return mu + std * epsilon             # z = μ + σε  (differentiable!)
        return mu   # deterministic at eval time

    def forward(self, x: torch.Tensor):
        mu, logvar = self.encoder(x)
        z          = self.reparameterize(mu, logvar)
        x_hat      = self.decoder(z)
        return x_hat, mu, logvar

    @torch.no_grad()
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Return only the mean μ (used for latent space visualisation)."""
        mu, _ = self.encoder(x)
        return mu
        

    @torch.no_grad()
    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder(z)

    @torch.no_grad()
    def sample(self, n: int, device=None) -> torch.Tensor:
        """Draw n images from the prior p(z) = N(0,I) and decode them."""
        device = device or next(self.parameters()).device
        z = torch.randn(n, self.latent_dim, device=device)
        return self.decode(z)


# ── Sanity check ──────────────────────────────────────────────────────────────
_vae = VAE(latent_dim=2)
_x   = torch.randn(4, 1, 28, 28)
_xhat, _mu, _logvar = _vae(_x)
print(f"x      : {_x.shape}")
print(f"x̂     : {_xhat.shape}")
print(f"μ      : {_mu.shape}    (mean of posterior)")
print(f"log σ² : {_logvar.shape} (log-variance of posterior)")
n_params = sum(p.numel() for p in _vae.parameters() if p.requires_grad)
print(f"Params  : {n_params:,}")

In [ ]:
"""
=============================================================
 Cell 16 — VAE loss function and training loop
=============================================================
"""

def vae_loss(x_hat: torch.Tensor, x: torch.Tensor,
             mu: torch.Tensor, logvar: torch.Tensor,
             beta: float = 1.0):
    """
    Compute the VAE ELBO loss (we minimise it, so all terms are positive).

    Loss = Reconstruction + β · KL

    Reconstruction (Binary Cross-Entropy):
        -E[log p(x|z)] = BCE(x̂, x)  summed over pixels, averaged over batch.
        BCE is preferred over MSE for Bernoulli-like (0/1) pixel distributions.

    KL Divergence (closed-form for Gaussians):
        KL(N(μ,σ²) ‖ N(0,I)) = -½ Σ_j (1 + log σ_j² - μ_j² - σ_j²)

    Parameters
    ----------
    x_hat   : reconstructed image,   shape (batch, 784)
    x       : original image,         shape (batch, 1, 28, 28) or (batch, 784)
    mu      : posterior mean,         shape (batch, latent_dim)
    logvar  : posterior log-variance, shape (batch, latent_dim)
    beta    : KL weight (1 = standard VAE, >1 = β-VAE)

    Returns
    -------
    total_loss, recon_loss, kl_loss  — all scalars
    """
    x_flat = x.flatten(start_dim=1)   # ensure (batch, 784)
    batch  = x.size(0)

    # Reconstruction: BCE summed over pixels, then averaged over batch
    # reduction='sum' then divide by batch is equivalent to 'mean' over samples
    recon = F.binary_cross_entropy(x_hat, x_flat, reduction="sum") / batch

    # KL: -½ Σ(1 + log σ² - μ² - σ²)
    # log σ² = logvar, σ² = logvar.exp()
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / batch

    return recon + beta * kl, recon, kl


def train_vae(
    model: VAE,
    train_loader,
    test_loader,
    n_epochs: int   = 30,
    lr:       float = 1e-3,
    beta:     float = 1.0,
    device          = DEVICE,
) -> dict:
    """
    Train the VAE with the ELBO loss.

    Tracks total, reconstruction, and KL components separately
    so we can verify the two terms balance correctly during training.
    """
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {"train_total": [], "train_recon": [], "train_kl": [],
               "test_total":  [], "test_recon":  [], "test_kl":  []}

    for epoch in range(1, n_epochs + 1):
        # ── Train ──────────────────────────────────────────────────────────────
        model.train()
        sums = [0.0, 0.0, 0.0]   # total, recon, kl

        for x, _ in tqdm(train_loader, desc=f"Epoch {epoch}/{n_epochs}", leave=False):
            x = x.to(device)
            x_hat, mu, logvar = model(x)

            total, recon, kl = vae_loss(x_hat, x, mu, logvar, beta=beta)

            optimizer.zero_grad()
            total.backward()
            # Gradient clipping: prevents exploding gradients during early training
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            sums[0] += total.item()
            sums[1] += recon.item()
            sums[2] += kl.item()

        n = len(train_loader)
        history["train_total"].append(sums[0] / n)
        history["train_recon"].append(sums[1] / n)
        history["train_kl"].append(sums[2] / n)

        # ── Evaluate ───────────────────────────────────────────────────────────
        model.eval()
        sums = [0.0, 0.0, 0.0]
        with torch.no_grad():
            for x, _ in test_loader:
                x = x.to(device)
                x_hat, mu, logvar = model(x)
                total, recon, kl  = vae_loss(x_hat, x, mu, logvar, beta=beta)
                sums[0] += total.item()
                sums[1] += recon.item()
                sums[2] += kl.item()
        n = len(test_loader)
        history["test_total"].append(sums[0] / n)
        history["test_recon"].append(sums[1] / n)
        history["test_kl"].append(sums[2] / n)

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d} | "
                  f"total={history['test_total'][-1]:.1f}  "
                  f"recon={history['test_recon'][-1]:.1f}  "
                  f"KL={history['test_kl'][-1]:.1f}")

    return history

print("VAE loss and training loop defined.")

In [ ]:
"""
=============================================================
 Cell 17 — Train VAE on MNIST
=============================================================
"""
set_seed(42)
vae = VAE(latent_dim=2, hidden_dim=256)

print("Training VAE (latent_dim=2) on MNIST …")
history_vae = train_vae(vae, mnist_train_loader, mnist_test_loader,
                         n_epochs=30, lr=1e-3, beta=1.0)

# ── Plot all three loss components ────────────────────────────────────────────
# Seeing Reconstruction and KL separately confirms the model is balancing them.
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, key, colour in zip(axes,
                            ["total", "recon", "kl"],
                            ["steelblue", "coral", "seagreen"]):
    ax.plot(history_vae[f"train_{key}"], label="train", color=colour, linewidth=2)
    ax.plot(history_vae[f"test_{key}"],  label="test",  color=colour,
            linewidth=2, linestyle="--")
    ax.set_title(f"VAE {key} loss")
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(alpha=0.3)
plt.suptitle("VAE training curves — MNIST", fontweight="bold")
plt.tight_layout()
plt.show()

# ── Reconstruction quality ────────────────────────────────────────────────────
x_test_batch, _ = next(iter(mnist_test_loader))
x_test_8 = x_test_batch[:8].to(DEVICE)

vae.eval()
with torch.no_grad():
    xhat_vae, _, _ = vae(x_test_8)
    xhat_vae = xhat_vae.reshape(-1, 1, 28, 28)

compare_rows(
    {"Original": x_test_8, "VAE Reconstruction": xhat_vae},
    n_images=8, img_shape=(1, 28, 28),
    suptitle="VAE reconstruction — MNIST test set",
)

# ── Generate new images by sampling from the prior p(z) = N(0,I) ─────────────
# This is the key difference from the classical AE: we can GENERATE new data!
generated = vae.sample(16).reshape(-1, 1, 28, 28)
imshow_grid(generated, suptitle="VAE generated samples (z ~ N(0,I), no input used)")

In [ ]:
"""
=============================================================
 Cell 18 — VAE latent space, grid traversal, and interpolation
=============================================================
The three key demos that showcase what makes a VAE special:

1. Coloured scatter: smooth, overlapping clusters (vs AE's sharp islands)
2. Grid traversal: every region of the latent space decodes to something reasonable
3. Class interpolation: morphing one digit into another through the latent space
"""

# ── 1. Latent space coloured by digit class ───────────────────────────────────
all_mu, all_y = [], []
vae.eval()
with torch.no_grad():
    for xb, yb in mnist_test_loader:
        mu = vae.encode(xb.to(DEVICE)).cpu()
        all_mu.append(mu)
        all_y.append(yb)

all_mu = torch.cat(all_mu).numpy()
all_y  = torch.cat(all_y).numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# VAE posterior means
sc = axes[0].scatter(all_mu[:, 0], all_mu[:, 1], c=all_y, cmap="tab10", s=4, alpha=0.7)
plt.colorbar(sc, ax=axes[0], ticks=range(10), label="Digit")
axes[0].set_title("VAE posterior means (μ)\nSmooth, overlapping clusters → continuous latent space")
axes[0].set_xlabel("z₁");  axes[0].set_ylabel("z₂")

# Compare: classical AE latent space (from Part 1)
axes[1].scatter(all_z[:, 0], all_z[:, 1], c=all_y, cmap="tab10", s=4, alpha=0.7)
axes[1].set_title("Classical AE latent space\nSharp, disjoint clusters → gaps = garbage when decoded")
axes[1].set_xlabel("z₁");  axes[1].set_ylabel("z₂")

plt.suptitle("Latent Space Comparison: VAE (left) vs Classical AE (right)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# ── 2. Grid traversal of the 2-D VAE latent space ────────────────────────────
# Every cell decodes to a realistic-looking image — there are no empty regions.
canvas = decode_grid(vae, grid_range=3.0, n=18)   # reuse helper from Part 1

plt.figure(figsize=(9, 9))
plt.imshow(canvas, cmap="gray", origin="upper")
plt.axis("off")
plt.title("VAE Latent Space Grid — every point decodes to a valid-looking digit\n"
          "(Compare to the classical AE grid: VAE has smoother transitions)", fontsize=10)
plt.tight_layout()
plt.show()

# ── 3. Smooth class-to-class interpolation ────────────────────────────────────
def interpolate_classes(vae, dataset, class_a: int, class_b: int,
                         n_steps: int = 12, device=DEVICE):
    """
    Compute mean latent codes μ_a and μ_b for two classes, then linearly
    interpolate between them: z(t) = (1-t)·μ_a + t·μ_b, t ∈ [0, 1].

    Decoding along this line shows a smooth morphing of one digit into another.
    """
    vae.eval()
    # Collect up to 200 examples per class
    imgs_a = torch.stack([x for x, y in dataset if y == class_a][:200])
    imgs_b = torch.stack([x for x, y in dataset if y == class_b][:200])

    with torch.no_grad():
        mu_a = vae.encode(imgs_a.to(device)).mean(dim=0, keepdim=True)  # (1, 2)
        mu_b = vae.encode(imgs_b.to(device)).mean(dim=0, keepdim=True)

        alphas = torch.linspace(0, 1, n_steps, device=device)

        frames = []
        for alpha in alphas:
            z = (1 - alpha) * mu_a + alpha * mu_b   # linear interpolation
            img = vae.decode(z).reshape(1, 28, 28)
            frames.append(img.cpu())

    fig, axes = plt.subplots(1, n_steps, figsize=(n_steps * 1.4, 2.0))
    for k, (frame, alpha) in enumerate(zip(frames, alphas)):
        axes[k].imshow(frame.squeeze().numpy().clip(0, 1), cmap="gray")
        axes[k].set_title(f"{alpha:.2f}", fontsize=7)
        axes[k].axis("off")

    axes[0].set_xlabel(f"Digit {class_a}", fontsize=9)
    axes[-1].set_xlabel(f"Digit {class_b}", fontsize=9)
    plt.suptitle(f"VAE Interpolation: {class_a} → {class_b}  "
                 f"(α=0 = pure class {class_a}, α=1 = pure class {class_b})",
                 fontsize=10, fontweight="bold")
    plt.tight_layout()
    plt.show()

# Demonstrate smooth transition for several digit pairs
for pair in [(0, 1), (3, 8), (4, 9)]:
    interpolate_classes(vae, mnist_train, *pair)

---
### 🏠 Homework — Part 3: Variational Autoencoder

#### Task A — FashionMNIST VAE
```python
set_seed(0)
vae_fmnist = VAE(latent_dim=2, hidden_dim=256)
history_vf  = train_vae(vae_fmnist, fmnist_train_loader, fmnist_test_loader,
                         n_epochs=30, beta=1.0)
# 1. Plot the 2-D latent space coloured by the 10 clothing categories.
# 2. Run decode_grid(vae_fmnist) — do the transitions look smoother than the AE?
# 3. Interpolate between a Sneaker (class 7) and an Ankle boot (class 9).
```

#### Task B — β-VAE experiment on MNIST
The β hyper-parameter controls the degree of disentanglement. Train three VAEs:

```python
for beta in [0.5, 1.0, 4.0]:
    set_seed(42)
    m = VAE(latent_dim=2, hidden_dim=256)
    train_vae(m, mnist_train_loader, mnist_test_loader, n_epochs=20, beta=beta)
    # Plot the latent space. How does increasing β affect:
    #   a) The KL loss (does the prior hold tighter)?
    #   b) Reconstruction sharpness?
    #   c) The structure of the 2-D scatter plot?
```

#### Task C — CIFAR-10 Convolutional VAE (advanced)
Combine the `ConvEncoder` / `ConvDecoder` from Part 1 with the VAE framework:
- Replace the flat fc_mu / fc_logvar heads with linear layers on top of `ConvEncoder`
- Use BCE or MSE reconstruction loss
- Visualise the 2-D grid traversal for CIFAR-10

#### Reflection questions
1. The VAE latent space looks more "circular" than the AE's. Why does the KL term cause this?
2. What is **posterior collapse**? When does it happen and what symptom do you see in the KL curve?
3. Why is binary cross-entropy used instead of MSE for the reconstruction term when pixels are in [0,1]?

---
## Part 4 — Sparse Autoencoder (SAE)

### The problem with dense representations

Even with a bottleneck, hidden neurons in a standard AE fire for many unrelated inputs — one neuron can encode "digit 3" and "bright background" simultaneously. This is called **polysemanticity** and it makes representations hard to interpret.

### Solution: Overcomplete + Sparse

A Sparse AE deliberately uses a **wider** hidden layer than the input ($d_{\text{hidden}} > d_{\text{input}}$, called *overcomplete*) but penalises the activations to keep most of them zero. With enough units, each neuron can specialise to exactly one feature.

### L1 penalty — the flat-rate tax

$$\mathcal{L}_{\text{total}} = \underbrace{\frac{1}{N}\sum\|x - \hat{x}\|^2}_{\text{Reconstruction}} + \lambda \underbrace{\frac{1}{N}\sum_i \|h_i\|_1}_{\text{Sparsity (L1)}}$$

Why L1 and not L2?

- **L2 (Ridge)** gradient = $2\lambda w$ — small weights shrink slowly, never exactly zero.  
- **L1 (Lasso)** gradient = $\pm\lambda$ — constant pull, pushes all weak activations to *exactly* zero.

Geometrically, L1 adds a *diamond* constraint (sharp corners on axes), while L2 adds a *sphere* (smooth surface). Optimal solutions are drawn to the corners of the diamond → many exact zeros.

### KL divergence penalty — average sparsity control (Homework)

An alternative: treat the average activation of neuron $j$ across a batch as a probability $\hat{\rho}_j$, and push it toward a target $\rho$ (e.g. 5%):

$$\Omega_{\text{KL}} = \beta \sum_j D_{\text{KL}}(\rho \,\|\, \hat{\rho}_j) = \beta \sum_j \left[\rho \log\frac{\rho}{\hat{\rho}_j} + (1-\rho)\log\frac{1-\rho}{1-\hat{\rho}_j}\right]$$

This forms an asymmetric U-shape centred at $\hat{\rho}_j = \rho$: neurons that fire too often OR never fire are both penalised heavily.

### What the learned features look like

The decoder weight matrix $W_{\text{dec}} \in \mathbb{R}^{784 \times d_{\text{hidden}}}$ contains one column per hidden neuron. Each column is a 784-dimensional vector that can be reshaped to a 28×28 image — revealing what *pattern* that neuron detects. With enough hidden units and sparsity, these patterns often look like Gabor filters or prototype digit strokes.

In [ ]:
"""
=============================================================
 Cell 21 — Sparse Autoencoder model + training with L1 penalty
=============================================================
Key design decision: SINGLE hidden layer, overcomplete.
Single layer makes the decoder weights directly visualisable
as a "dictionary" of learned feature atoms.
"""

class SparseAE(nn.Module):
    """
    Single-hidden-layer, overcomplete Sparse Autoencoder.

    Architecture:
        input_dim  →  hidden_dim  →  input_dim
          784            1024            784

    The hidden_dim > input_dim means the network has MORE features available
    than input pixels — it is overcomplete.  Without regularisation it would
    learn a trivial identity mapping.  With L1, most hidden units stay silent
    for any given input, and each fires only for its specific pattern.

    Activation choice:
      • ReLU forces exact zeros for negative pre-activations (natural sparsity)
      • Sigmoid (used for KL variant) gives continuous values in (0,1)
    """
    def __init__(self, input_dim: int = 784, hidden_dim: int = 1024):
        super().__init__()
        self.hidden_dim = hidden_dim

        # Encoder: linear → ReLU (ReLU contributes to sparsity by zeroing negatives)
        self.enc_linear = nn.Linear(input_dim, hidden_dim, bias=True)
        self.activation  = nn.ReLU()

        # Decoder: single linear layer; its weight columns are the dictionary atoms.
        # No BatchNorm here — we want to visualise raw weights without normalisation.
        self.dec_linear = nn.Linear(hidden_dim, input_dim, bias=True)
        self.out_act     = nn.Sigmoid()

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Return hidden activations h (before output sigmoid)."""
        return self.activation(self.enc_linear(x.flatten(start_dim=1)))

    def decode(self, h: torch.Tensor) -> torch.Tensor:
        return self.out_act(self.dec_linear(h))

    def forward(self, x: torch.Tensor):
        h     = self.encode(x)
        x_hat = self.decode(h)
        return x_hat, h   # return both so caller can compute sparsity loss


def train_sparse_ae(
    model: SparseAE,
    train_loader,
    test_loader,
    n_epochs: int   = 35,
    lr:       float = 1e-3,
    lambda_l1: float = 5e-4,   # sparsity strength; tune: larger = more sparse
    device          = DEVICE,
) -> dict:
    """
    Train Sparse AE with L1 regularisation on hidden activations.

    Total loss = MSE(x, x̂) + λ · mean(|h|)

      • MSE(x, x̂)    → reconstruction fidelity
      • λ · mean(|h|) → sparsity: penalise each non-zero activation equally

    h.abs().mean() is averaged over both the batch AND hidden dimensions,
    giving a per-pixel-per-neuron normalised scale that does not explode
    with large hidden_dim.
    """
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {"train_total": [], "train_recon": [], "train_l1": [],
               "test_recon":  []}

    for epoch in range(1, n_epochs + 1):
        model.train()
        t_tot = t_rec = t_l1 = 0.0

        for x, _ in tqdm(train_loader, desc=f"Epoch {epoch}/{n_epochs}", leave=False):
            x     = x.to(device)
            x_hat, h = model(x)
            x_tgt = x.flatten(start_dim=1)

            recon_loss = F.mse_loss(x_hat, x_tgt)
            # L1 sparsity: encourage most hidden activations to be exactly zero
            l1_loss    = lambda_l1 * h.abs().mean()
            total_loss = recon_loss + l1_loss

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

            t_tot += total_loss.item()
            t_rec += recon_loss.item()
            t_l1  += l1_loss.item()

        n = len(train_loader)
        history["train_total"].append(t_tot / n)
        history["train_recon"].append(t_rec / n)
        history["train_l1"].append(t_l1 / n)

        # Test reconstruction (no L1 at test time — it's a training regulariser)
        model.eval()
        te_rec = 0.0
        with torch.no_grad():
            for x, _ in test_loader:
                x = x.to(device)
                x_hat, _ = model(x)
                te_rec += F.mse_loss(x_hat, x.flatten(start_dim=1)).item()
        history["test_recon"].append(te_rec / len(test_loader))

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d} | total={history['train_total'][-1]:.5f}  "
                  f"recon={history['train_recon'][-1]:.5f}  "
                  f"L1={history['train_l1'][-1]:.5f}  "
                  f"test_recon={history['test_recon'][-1]:.5f}")

    return history

print("SparseAE and train_sparse_ae() defined.")

In [ ]:
"""
=============================================================
 Cell 22 — Train Sparse AE + compare λ values
=============================================================
We train three models with different λ (L1 strength) to show the
sparsity-reconstruction trade-off.
"""

results_sparse = {}

for lam in [0.0, 1e-4, 5e-4]:
    label = f"λ={lam}"
    print(f"\n{'='*50}")
    print(f"Training SAE with {label}  (hidden_dim=1024)")
    print(f"{'='*50}")
    set_seed(42)
    m = SparseAE(input_dim=784, hidden_dim=1024)
    h = train_sparse_ae(m, mnist_train_loader, mnist_test_loader,
                        n_epochs=35, lr=1e-3, lambda_l1=lam)
    results_sparse[label] = {"model": m, "history": h}

# ── Loss curves ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
colours = ["steelblue", "coral", "seagreen"]

for (label, res), col in zip(results_sparse.items(), colours):
    axes[0].plot(res["history"]["train_recon"], color=col, label=label, linewidth=2)
    axes[1].plot(res["history"]["train_l1"],    color=col, label=label, linewidth=2)

for ax, title in zip(axes, ["Reconstruction loss (MSE)", "L1 sparsity penalty"]):
    ax.set_title(title);  ax.set_xlabel("Epoch");  ax.legend();  ax.grid(alpha=0.3)

plt.suptitle("SAE: Effect of L1 strength λ on MNIST", fontweight="bold")
plt.tight_layout()
plt.show()

# ── Reconstruction quality at each λ ─────────────────────────────────────────
x_test_8 = next(iter(mnist_test_loader))[0][:8].to(DEVICE)

rows = {"Original": x_test_8}
for label, res in results_sparse.items():
    res["model"].eval()
    with torch.no_grad():
        xhat, _ = res["model"](x_test_8)
    rows[f"SAE ({label})"] = xhat.reshape(-1, 1, 28, 28)

compare_rows(rows, n_images=8, img_shape=(1, 28, 28),
             suptitle="Reconstruction quality at different λ values (larger λ = more sparse, possibly blurrier)")

In [ ]:
"""
=============================================================
 Cell 23 — Sparsity analysis + dictionary atom visualisation
=============================================================
We use the best SAE (λ=5e-4) for both analyses.
"""

sae = results_sparse["λ=0.0005"]["model"]   # trained with λ=5e-4
sae.eval()

# ── 1. Sparsity statistics ────────────────────────────────────────────────────
print("Computing sparsity statistics over the MNIST test set …")
all_h = []
with torch.no_grad():
    for x, _ in mnist_test_loader:
        _, h = sae(x.to(DEVICE))
        all_h.append(h.cpu())

all_h = torch.cat(all_h, dim=0)   # (10000, 1024)

# Fraction of EXACT zeros (from ReLU — not just small values)
sparsity         = (all_h == 0).float().mean().item()
active_per_sample = (all_h > 0).float().sum(dim=1).mean().item()
dead_neurons     = (all_h.max(dim=0).values == 0).sum().item()

print(f"\n  Hidden dimension      : {sae.hidden_dim}")
print(f"  Sparsity (frac zeros) : {sparsity:.3f}  = {sparsity*100:.1f}% of activations are 0")
print(f"  Active neurons/sample : {active_per_sample:.1f} / {sae.hidden_dim}")
print(f"  Dead neurons (never)  : {dead_neurons}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))

# Histogram of non-zero activations — should be heavily skewed toward 0
vals = all_h.numpy().flatten()
ax1.hist(vals[vals > 0], bins=80, color="steelblue", edgecolor="white", linewidth=0.3)
ax1.set_title(f"Non-zero activation distribution\n"
              f"Sparsity = {sparsity:.1%}  (most activations are exactly 0)")
ax1.set_xlabel("Activation value");  ax1.set_ylabel("Count (log)");  ax1.set_yscale("log")
ax1.grid(alpha=0.3)

# Per-neuron firing rate (how often each neuron fires across all test images)
firing_rate = (all_h > 0).float().mean(dim=0).numpy()   # (1024,)
ax2.hist(firing_rate, bins=50, color="coral", edgecolor="white", linewidth=0.3)
ax2.set_title("Neuron firing rate distribution\n"
              "(ideal: peak near left = most neurons fire rarely)")
ax2.set_xlabel("Fraction of inputs where neuron fires")
ax2.set_ylabel("Number of neurons")
ax2.grid(alpha=0.3)

plt.suptitle("SAE Sparsity Analysis (λ=5×10⁻⁴, hidden_dim=1024)", fontweight="bold")
plt.tight_layout()
plt.show()

# ── 2. Dictionary atoms (decoder weights reshaped to 28×28) ──────────────────
# W_dec shape: (784, 1024)  — column j is the "image template" for neuron j.
# We select the 64 neurons with the highest average activation (most informative).
W_dec = sae.dec_linear.weight.detach().cpu()   # (784, 1024)

# Rank neurons by average firing rate
top_indices = firing_rate.argsort()[::-1][:64]   # 64 most active neurons

n_show = 64
n_cols = 8
n_rows = n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.6, n_rows * 1.6))
axes = axes.flatten()

for i, neuron_idx in enumerate(top_indices):
    atom = W_dec[:, neuron_idx].numpy()           # (784,) decoder column
    atom = atom.reshape(28, 28)                   # reshape to image
    # Normalise to [0,1] for display (atoms can have negative weights)
    atom = (atom - atom.min()) / (atom.max() - atom.min() + 1e-8)
    axes[i].imshow(atom, cmap="RdBu_r")
    axes[i].set_title(f"#{neuron_idx}\n{firing_rate[neuron_idx]:.2f}", fontsize=6)
    axes[i].axis("off")

plt.suptitle(
    "SAE Dictionary Atoms — top 64 most-active neurons\n"
    "Each tile = what pattern that neuron detects (decoder weight column)\n"
    "RdBu_r: red=positive weight, blue=negative weight",
    fontsize=10, fontweight="bold"
)
plt.tight_layout()
plt.show()

---
### 🏠 Homework — Part 4: Sparse Autoencoder

#### Task A — FashionMNIST & CIFAR-10 with L1 penalty

```python
# FashionMNIST SAE (same 784-dim input)
set_seed(0)
sae_fmnist = SparseAE(input_dim=784, hidden_dim=1024)
train_sparse_ae(sae_fmnist, fmnist_train_loader, fmnist_test_loader,
                n_epochs=20, lambda_l1=5e-4)
# Visualise dictionary atoms — do they look like clothing textures?

# CIFAR-10 SAE (3072-dim flat input — alternative to ConvAE)
set_seed(0)
sae_cifar = SparseAE(input_dim=3072, hidden_dim=4096)
# Write a loader that flattens CIFAR batches and call train_sparse_ae.
# Visualise atoms reshaped to (3, 32, 32) as RGB images.
```

#### Task B — KL Divergence Sparsity Penalty

Complete the implementation below and compare it against the L1 model:

```python
def kl_sparsity_penalty(h: torch.Tensor, rho: float = 0.05,
                         eps: float = 1e-8) -> torch.Tensor:
    """
    KL divergence sparsity penalty.

    Treat each neuron's average activation across the batch as a
    probability rho_hat_j.  Penalise deviation from target rho.

    KL(rho || rho_hat_j) = rho*log(rho/rho_hat_j) + (1-rho)*log((1-rho)/(1-rho_hat_j))

    Parameters
    ----------
    h   : hidden activations,  shape (batch, hidden_dim), values in (0, 1)
          Use Sigmoid activation (not ReLU) in the encoder for this variant.
    rho : target sparsity (e.g. 0.05 = 5% average firing rate)
    eps : small constant to avoid log(0)

    Returns
    -------
    KL divergence penalty scalar (sum over neurons, averaged implicitly by beta)
    """
    rho_hat = h.mean(dim=0).clamp(eps, 1.0 - eps)   # (hidden_dim,)
    # ← Your code here!  Implement the formula above using rho and rho_hat.
    kl = ...
    return kl.sum()


class SparseAE_KL(nn.Module):
    """
    Sparse AE with Sigmoid hidden activations (needed for KL variant).
    Architecture identical to SparseAE, but ReLU → Sigmoid in the encoder.
    """
    def __init__(self, input_dim=784, hidden_dim=1024):
        super().__init__()
        self.enc_linear = nn.Linear(input_dim, hidden_dim)
        self.activation  = nn.Sigmoid()   # ← keeps activations in (0,1) for KL
        self.dec_linear = nn.Linear(hidden_dim, input_dim)
        self.out_act     = nn.Sigmoid()

    def forward(self, x):
        h = self.activation(self.enc_linear(x.flatten(1)))
        return self.out_act(self.dec_linear(h)), h


# Training loop for KL variant:
def train_sae_kl(model, train_loader, test_loader,
                  n_epochs=20, lr=1e-3, beta=0.5, rho=0.05, device=DEVICE):
    model = model.to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(1, n_epochs+1):
        model.train()
        for x, _ in train_loader:
            x = x.to(device)
            x_hat, h = model(x)
            recon = F.mse_loss(x_hat, x.flatten(1))
            kl    = beta * kl_sparsity_penalty(h, rho=rho)   # ← your function
            loss  = recon + kl
            opt.zero_grad(); loss.backward(); opt.step()
        if epoch % 5 == 0:
            print(f"Epoch {epoch:3d} | loss={loss.item():.5f}")
    return model
```

#### Task C — λ sensitivity experiment (FashionMNIST)
Train four SAEs with λ ∈ {0, 1e-5, 1e-4, 1e-3}. For each, record:
1. Test MSE
2. % of zero activations (sparsity)
3. Number of dead neurons

Plot all three metrics vs λ on the same figure.

#### Reflection questions
1. Why does the KL penalty prevent both dead neurons AND hyperactive neurons, while L1 only prevents hyperactive ones?
2. In mechanistic interpretability of LLMs, SAEs are applied to the *activations of a pre-trained model*, not trained from scratch. Why does this make sense? What would the dictionary atoms correspond to?
3. If you increase `hidden_dim` from 1024 → 4096 with the same λ, does sparsity increase, decrease, or stay the same? Why?